## 01 — Writing the LOD Files

We have a design. Now we execute it.

This notebook:

1. Defines the four LOD configurations
2. Implements the simplification pipeline
3. Writes four GeoJSON files to `data/lod/`
4. Reports what was produced

Run this notebook once. The output files are used by every module that follows.

## Setup

In [1]:
import json
import time
from pathlib import Path
from shapely.geometry import LineString

data_path  = Path("../../data/ne_10m_railroads.geojson")
output_dir = Path("../../data/lod")
output_dir.mkdir(parents=True, exist_ok=True)

with open(data_path) as f:
    railroads = json.load(f)

features = railroads["features"]
print(f"Loaded {len(features):,} features")
print(f"Output directory: {output_dir.resolve()}")

Loaded 25,413 features
Output directory: /workspaces/CMPS-4543-Salyers/Assignments/03-Data_Manager/data/lod


## LOD Configuration

Each level is defined by three parameters:
- `epsilon` — the simplification tolerance in degrees
- `max_scalerank` — maximum scalerank to include (`None` = include all)
- `zoom_range` — informational, used in the output properties

In [2]:
LOD_LEVELS = [
    {
        "name":          "coarse",
        "filename":      "railroads_coarse.geojson",
        "epsilon":       1.0,
        "max_scalerank": 4,
        "zoom_range":    "1-3",
    },
    {
        "name":          "medium",
        "filename":      "railroads_medium.geojson",
        "epsilon":       0.1,
        "max_scalerank": None,
        "zoom_range":    "4-6",
    },
    {
        "name":          "fine",
        "filename":      "railroads_fine.geojson",
        "epsilon":       0.01,
        "max_scalerank": None,
        "zoom_range":    "7-10",
    },
    {
        "name":          "extra_fine",
        "filename":      "railroads_extra_fine.geojson",
        "epsilon":       0.001,
        "max_scalerank": None,
        "zoom_range":    "11+",
    },
]

## The Simplification Function

One function handles a single feature at a given epsilon. It returns the simplified feature dict, or `None` if the feature collapses.

In [3]:
def simplify_feature(feature, epsilon):
    """
    Simplify one GeoJSON LineString feature using Shapely's Douglas-Peucker.
    Returns a new feature dict, or None if the result has fewer than 2 points.
    """
    coords = feature["geometry"]["coordinates"]

    simplified_geom = LineString(coords).simplify(epsilon, preserve_topology=False)
    simplified_coords = list(simplified_geom.coords)

    if len(simplified_coords) < 2:
        return None

    return {
        "type":       "Feature",
        "properties": feature["properties"],
        "geometry":   {
            "type":        "LineString",
            "coordinates": simplified_coords,
        },
    }

## The Pipeline

For each LOD level:
1. Filter by `max_scalerank` if set
2. Simplify each feature
3. Drop `None` results (collapsed features)
4. Write to a GeoJSON file

In [4]:
print(f"{'Level':<12} {'Epsilon':>8} {'In':>8} {'Out':>8} {'Dropped':>9} {'Size (MB)':>11} {'Time (s)':>10}")
print("-" * 72)

for level in LOD_LEVELS:
    t0 = time.perf_counter()

    # Step 1 — feature filter
    if level["max_scalerank"] is not None:
        candidates = [
            f for f in features
            if f["properties"]["scalerank"] <= level["max_scalerank"]
        ]
    else:
        candidates = features

    # Step 2 & 3 — simplify and drop collapsed
    simplified = []
    for f in candidates:
        result = simplify_feature(f, level["epsilon"])
        if result is not None:
            simplified.append(result)

    # Step 4 — write output
    collection = {"type": "FeatureCollection", "features": simplified}
    out_path = output_dir / level["filename"]

    with open(out_path, "w") as f:
        json.dump(collection, f)

    elapsed   = time.perf_counter() - t0
    size_mb   = out_path.stat().st_size / 1_000_000
    dropped   = len(candidates) - len(simplified)

    print(
        f"{level['name']:<12} {level['epsilon']:>8} "
        f"{len(candidates):>8,} {len(simplified):>8,} "
        f"{dropped:>9,} {size_mb:>10.2f} {elapsed:>10.2f}"
    )

print("-" * 72)
print(f"\nAll files written to: {output_dir.resolve()}")

Level         Epsilon       In      Out   Dropped   Size (MB)   Time (s)
------------------------------------------------------------------------
coarse            1.0    2,845    2,845         0       1.08       0.34
medium            0.1   25,413   25,413         0       9.66       2.07
fine             0.01   25,413   25,413         0      11.34       2.70
extra_fine      0.001   25,413   25,413         0      18.98       3.95
------------------------------------------------------------------------

All files written to: /workspaces/CMPS-4543-Salyers/Assignments/03-Data_Manager/data/lod


## Verifying the Output Files

Let's confirm the files exist and are valid GeoJSON by loading one back.

In [5]:
print("Output files:")
for level in LOD_LEVELS:
    path = output_dir / level["filename"]
    size_mb = path.stat().st_size / 1_000_000
    print(f"  {level['filename']:<40} {size_mb:.2f} MB")

print()

# Spot-check: load the fine level and verify structure
with open(output_dir / "railroads_fine.geojson") as f:
    check = json.load(f)

print(f"Fine level spot-check:")
print(f"  type:          {check['type']}")
print(f"  feature count: {len(check['features']):,}")
print(f"  first feature geometry type: {check['features'][0]['geometry']['type']}")
print(f"  first feature coord count:   {len(check['features'][0]['geometry']['coordinates'])}")

Output files:
  railroads_coarse.geojson                 1.08 MB
  railroads_medium.geojson                 9.66 MB
  railroads_fine.geojson                   11.34 MB
  railroads_extra_fine.geojson             18.98 MB

Fine level spot-check:
  type:          FeatureCollection
  feature count: 25,413
  first feature geometry type: LineString
  first feature coord count:   2


## Exercise A

The coarse level applies a `scalerank <= 4` filter before simplification. Modify the pipeline (in a copy below) to run the coarse level **without** the scalerank filter.

Compare:
- How many features does the unfiltered coarse level contain?
- How much larger is the file?
- Is the scalerank filter worth keeping, or is geometry simplification alone sufficient?

Write your conclusion as a comment.

In [6]:
# Run coarse simplification without the scalerank filter
print(f"{'Level':<12} {'Epsilon':>8} {'In':>8} {'Out':>8} {'Dropped':>9} {'Size (MB)':>11} {'Time (s)':>10}")
print("-" * 72)

level = dict(LOD_LEVELS[0])  # copy the coarse level
level["max_scalerank"] = None  # disable scalerank filter
t0 = time.perf_counter()

# Step 1 — feature filter
if level["max_scalerank"] is not None:
    candidates = [
        f for f in features
        if f["properties"]["scalerank"] <= level["max_scalerank"]
    ]
else:
    candidates = features

# Step 2 & 3 — simplify and drop collapsed
simplified = []
for f in candidates:
    result = simplify_feature(f, level["epsilon"])
    if result is not None:
        simplified.append(result)

# Step 4 — write output
collection = {"type": "FeatureCollection", "features": simplified}
out_path = output_dir / level["filename"]

with open(out_path, "w") as f:
    json.dump(collection, f)

elapsed   = time.perf_counter() - t0
size_mb   = out_path.stat().st_size / 1_000_000
dropped   = len(candidates) - len(simplified)

print(
    f"{level['name']:<12} {level['epsilon']:>8} "
    f"{len(candidates):>8,} {len(simplified):>8,} "
    f"{dropped:>9,} {size_mb:>10.2f} {elapsed:>10.2f}"
)

print("-" * 72)
print(f"\nAll files written to: {output_dir.resolve()}")

Level         Epsilon       In      Out   Dropped   Size (MB)   Time (s)
------------------------------------------------------------------------


coarse            1.0   25,413   25,413         0       9.55       2.06
------------------------------------------------------------------------

All files written to: /workspaces/CMPS-4543-Salyers/Assignments/03-Data_Manager/data/lod


## Exercise B

The pipeline writes GeoJSON using `json.dump()`, which produces unformatted output (no indentation). This is intentional — indentation adds whitespace that inflates file size.

1. Write one of the LOD files again with `indent=2` to make it human-readable.
2. Compare the file size before and after.
3. By what percentage does indentation increase file size?

Do not keep the indented file — overwrite it with the compact version when you are done.

In [7]:
# Write a LOD file with indent=2, measure the size difference, then restore compact version
# Your code here
indented_path = output_dir / "railroads_coarse_indented.geojson"
with open(indented_path, "w") as f:
    json.dump(collection, f, indent = 2)
indented_size_mb = indented_path.stat().st_size / 1_000_000
print(f"Indented coarse file size: {indented_size_mb:.2f} MB")
# Restore compact version
indented_path.unlink()
print(f"Restored compact version of coarse file: {indented_size_mb:.2f} MB")


Indented coarse file size: 16.11 MB
Restored compact version of coarse file: 16.11 MB


## Check Your Understanding

We use `shapely.simplify(epsilon, preserve_topology=False)`.

Shapely also has `preserve_topology=True`. Look up what that parameter does, then answer:

For a railroad LOD pipeline, should we use `True` or `False`? Why?

Write a 2–3 sentence answer.

Preserve topology makes sure that the geometry is valid compared to the raw data. Making this false prioritizes speed and simplification over validity.

---

## Next

In [02 — Comparing the Levels](./02-Comparing_the_Levels.ipynb), we load all four output files and compare them visually on a map.